
# **Chapter 1: Common Conventions and API Elements of scikit-learn**

**Buku:** scikit-learn Cookbook (O'Reilly)

**Topik:** Konvensi umum dan elemen API scikit-learn

## **Ringkasan Chapter**

Kekuatan scikit-learn bukan pada satu algoritma tertentu, tetapi pada desain API yang seragam. Semua model dan alat preprocessing memakai pola method yang sama, sehingga setelah paham polanya satu kali, kita bisa memakainya untuk ratusan algoritma lain.

Chapter ini membahas:

1. **Estimator** (`fit()`, `predict()`, `fit_predict()`): objek yang belajar dari data.
2. **Transformer** (`fit()`, `transform()`, `fit_transform()`): objek yang mengubah data.
3. **Custom estimator/transformer**: membuat komponen sendiri dengan `BaseEstimator` dan mixin.
4. **Pipeline**: merangkai beberapa langkah menjadi satu alur kerja.
5. **Atribut dan method umum**: `coef_`, `intercept_`, `score()`.
6. **Hyperparameter tuning**: `get_params()`, `set_params()`, `GridSearchCV`, `RandomizedSearchCV`.
7. **Metadata**: estimator tags dan metadata routing.

> Cara pakai notebook ini: jalankan sel dari atas ke bawah. Setiap sel kode diikuti penjelasan teori dan cara membaca hasilnya.

In [7]:
import numpy as np
import sklearn

print("scikit-learn version:", sklearn.__version__)
print("NumPy version       :", np.__version__)

scikit-learn version: 1.6.1
NumPy version       : 2.1.3


**Penjelasan:** sel ini hanya mengecek versi library. Cek versi penting karena beberapa fitur di chapter ini (misalnya `__sklearn_tags__()` dan metadata routing) hanya tersedia di scikit-learn versi baru (sekitar 1.6 ke atas). Di Google Colab, library ini sudah terpasang.

## **1. Filosofi Desain scikit-learn**

scikit-learn dibangun di atas empat prinsip:

| Prinsip | Arti |
|:--|:--|
| **Consistency** | Semua objek memakai antarmuka yang sama (`fit`, `predict`, `transform`). |
| **Simplicity** | Hyperparameter punya nilai default yang masuk akal, jadi bisa langsung dipakai. |
| **Modularity** | Estimator, transformer, dan pipeline adalah blok bangunan yang berdiri sendiri. |
| **Reusability** | Komponen yang dibuat untuk satu proyek bisa dipakai di proyek lain. |

Akibat praktisnya: mengganti algoritma hanya perlu mengubah satu baris. Contoh, dari `LogisticRegression()` ke `RandomForestClassifier()`, sedangkan kode `fit`/`predict` di bawahnya tetap sama.

## **2. Estimator: `fit()` dan `predict()`**

**Estimator** adalah objek apa pun yang bisa belajar dari data. Minimal ia punya method `fit()`.

Pada **supervised learning**, kita punya matriks fitur $\mathbf{X} \in \mathbb{R}^{n \times p}$ ($n$ sampel, $p$ fitur) dan target $\mathbf{y}$. `fit()` mempelajari pemetaan $\hat{f}: \mathbf{X} \rightarrow \mathbf{y}$, lalu `predict()` memakainya pada data baru.


---


### **Contoh: Linear Regression**

Linear regression mencari bobot $\mathbf{w}$ dan bias $b$ yang meminimalkan jumlah kuadrat residual:

$$\min_{\mathbf{w}, b} \sum_{i=1}^{n} \left( y_i - (\mathbf{w}^T \mathbf{x}_i + b) \right)^2$$

Solusi tertutupnya (*normal equations*):

$$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

In [8]:
from sklearn.linear_model import LinearRegression

# Data contoh: 5 sampel, 1 fitur
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

# Pola 3 langkah: buat -> fit -> predict
model = LinearRegression()
model.fit(X, y)

X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print("Prediksi untuk X = [6, 7]:", predictions)

Prediksi untuk X = [6, 7]: [5.75 6.7 ]


**Penjelasan hasil:** model memprediksi sekitar **5.75** untuk $X=6$ dan **6.70** untuk $X=7$. Angka ini mengikuti garis $\hat{y} = 0.95x + 0.05$ yang dipelajari saat `fit()`.

Pola **buat objek, `fit`, lalu `predict`** sama untuk semua estimator supervised di scikit-learn, mulai dari regresi linear sampai gradient boosting.

### `fit_predict()`: jalan pintas untuk unsupervised learning

Pada **unsupervised learning** (misalnya clustering) tidak ada target $\mathbf{y}$. Yang kita inginkan biasanya label cluster untuk data yang sama dengan data latih, sehingga `fit()` dan `predict()` bisa digabung menjadi `fit_predict()`.

**K-Means** membagi data ke dalam $K$ cluster dengan meminimalkan jarak kuadrat ke pusat cluster:

$$\min_{\{C_k\}} \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

dengan $\boldsymbol{\mu}_k$ adalah rata-rata (centroid) cluster $C_k$.

In [9]:
from sklearn.cluster import KMeans

X = np.array([[1], [2], [3], [4], [5]])

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

print("Label cluster :", labels)
print("Pusat cluster :", kmeans.cluster_centers_.flatten())

Label cluster : [0 0 0 1 1]
Pusat cluster : [2.  4.5]


**Penjelasan hasil:** lima titik terbagi menjadi dua kelompok. Label `0` dan `1` hanya penanda grup, bukan urutan atau ranking. Perhatikan bahwa pusat cluster adalah rata-rata anggotanya: jika anggota cluster adalah `[1, 2, 3]` maka pusatnya `2.0`, dan jika `[4, 5]` maka pusatnya `4.5`. Nomor label yang muncul bisa berbeda di setiap mesin, jadi yang penting adalah pengelompokannya.

**Kapan memakai yang mana?**
- **Supervised:** `fit(X_train, y_train)` lalu `predict(X_test)` secara terpisah. Data latih dan uji harus tetap terpisah.
- **Unsupervised:** `fit_predict(X)` praktis bila kita hanya ingin label untuk data latih itu sendiri.

## 3. Transformer: `fit()`, `transform()`, `fit_transform()`

**Transformer** adalah estimator yang **mengubah data**, bukan membuat prediksi.

- `fit(X)`: mempelajari parameter dari data latih (misalnya mean dan standar deviasi).
- `transform(X)`: menerapkan transformasi yang sudah dipelajari.
- `fit_transform(X)`: gabungan keduanya dalam satu panggilan.

### StandardScaler (standarisasi z-score)

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

Setiap fitur $j$ dipusatkan ke rata-rata 0 dan standar deviasi 1.

**Mengapa perlu?** Algoritma seperti SVM, regresi logistik, dan neural network sensitif terhadap skala fitur. Fitur berskala ribuan (misalnya pendapatan) akan mendominasi fitur berskala desimal (misalnya IPK) bila tidak diskalakan.

In [10]:
from sklearn.preprocessing import StandardScaler

X = np.array([[1, 2], [3, 4], [5, 6]])

scaler = StandardScaler()
scaler.fit(X)                      # belajar mean dan std
X_scaled = scaler.transform(X)     # terapkan transformasi

print("Data asli:\n", X)
print("\nData terskala:\n", X_scaled)
print("\nMean tiap fitur (hasil belajar):", scaler.mean_)
print("Std tiap fitur (hasil belajar) :", scaler.scale_)

Data asli:
 [[1 2]
 [3 4]
 [5 6]]

Data terskala:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Mean tiap fitur (hasil belajar): [3. 4.]
Std tiap fitur (hasil belajar) : [1.63299316 1.63299316]


**Penjelasan hasil:** kedua fitur punya mean **3.0** dan standar deviasi sekitar **1.633**. Contoh perhitungan manual untuk nilai pertama:

$$z = \frac{1 - 3.0}{1.633} \approx -1.2247$$

Hasilnya cocok dengan output. Karena data asli `[1, 3, 5]` berjarak sama, hasil skala simetris: `[-1.2247, 0, 1.2247]`.

**Aturan penting:** pakai `fit_transform()` pada **data latih**, tetapi hanya `transform()` pada **data uji**. Jika `fit()` dipanggil lagi pada data uji, mean dan std dihitung ulang dari data uji, sehingga terjadi **data leakage** dan skala data latih dan uji tidak lagi sama.

In [11]:
# Pola standar: fit_transform pada data latih
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Hasil fit_transform:\n", X_scaled)
print("\nMean kolom setelah skala:", X_scaled.mean(axis=0))
print("Std kolom setelah skala :", X_scaled.std(axis=0))

Hasil fit_transform:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Mean kolom setelah skala: [0. 0.]
Std kolom setelah skala : [1. 1.]


**Penjelasan hasil:** hasilnya identik dengan `fit()` lalu `transform()` terpisah, hanya lebih ringkas. Setelah diskalakan, mean tiap kolom sangat dekat dengan 0 (bisa muncul angka seperti `1e-17` karena pembulatan floating point) dan std tepat 1.

In [12]:
# Demonstrasi pola yang BENAR untuk train/test
from sklearn.model_selection import train_test_split

rng = np.random.RandomState(0)
X_all = rng.normal(loc=50, scale=10, size=(100, 2))
X_train, X_test = train_test_split(X_all, test_size=0.3, random_state=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit hanya di data latih
X_test_s  = scaler.transform(X_test)        # test hanya di-transform

print("Mean data latih (terskala):", X_train_s.mean(axis=0).round(4))
print("Mean data uji   (terskala):", X_test_s.mean(axis=0).round(4))

Mean data latih (terskala): [0. 0.]
Mean data uji   (terskala): [ 0.1369 -0.1346]


**Penjelasan hasil:** mean data latih tepat 0, sedangkan mean data uji hanya *mendekati* 0. Itu wajar dan memang benar: data uji memakai statistik dari data latih. Kalau mean data uji juga dipaksa menjadi 0 persis, berarti kita melakukan `fit` pada data uji (leakage).

## 4. Custom Estimator dan Transformer

API scikit-learn bisa diperluas. Dengan mengikuti konvensinya, kelas buatan kita otomatis kompatibel dengan `Pipeline`, `GridSearchCV`, dan `cross_val_score`.

Caranya: turunkan dari `BaseEstimator` ditambah **mixin**:

| Mixin | Fungsi |
|:--|:--|
| `ClassifierMixin` | Menambah `score()` default berupa akurasi |
| `RegressorMixin` | Menambah `score()` default berupa $R^2$ |
| `TransformerMixin` | Menambah `fit_transform()` otomatis |

*Mixin* adalah pola OOP Python untuk "menambahkan" kemampuan tertentu ke kelas tanpa membuat pohon pewarisan yang dalam.

Custom transformer wajib punya `fit(self, X, y=None)` (mengembalikan `self`) dan `transform(self, X)`.

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class ValueClipper(BaseEstimator, TransformerMixin):
    """Memotong semua nilai fitur ke rentang [lower, upper]."""

    def __init__(self, lower=0.0, upper=1.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.n_features_in_ = X.shape[1]
        self.is_fitted_ = True
        return self                       # penting: kembalikan self

    def transform(self, X):
        check_is_fitted(self, "is_fitted_")
        return np.clip(np.asarray(X), self.lower, self.upper)

X = np.array([[0.5, -0.3], [1.5, 0.7], [0.2, 2.1]])
clipper = ValueClipper(lower=0.0, upper=1.0)
clipper.fit(X)

print("Data asli:\n", X)
print("\nSetelah clipping [0, 1]:\n", clipper.transform(X))
print("\nfit_transform (gratis dari mixin):\n", ValueClipper().fit_transform(X))
print("\nParameter:", clipper.get_params())

Data asli:
 [[ 0.5 -0.3]
 [ 1.5  0.7]
 [ 0.2  2.1]]

Setelah clipping [0, 1]:
 [[0.5 0. ]
 [1.  0.7]
 [0.2 1. ]]

fit_transform (gratis dari mixin):
 [[0.5 0. ]
 [1.  0.7]
 [0.2 1. ]]

Parameter: {'lower': 0.0, 'upper': 1.0}


**Penjelasan hasil:** nilai `-0.3` menjadi `0.0`, serta `1.5` dan `2.1` menjadi `1.0`. Nilai yang sudah berada dalam rentang tidak berubah.

Aturan desain yang terlihat dari kode:

1. **`__init__` hanya menyimpan hyperparameter**, tanpa komputasi. Dengan begitu `get_params()` dan `set_params()` bekerja otomatis lewat `BaseEstimator`.
2. **`fit()` mengembalikan `self`**, sehingga bisa di-*chain*: `clipper.fit(X).transform(X)`.
3. **`check_is_fitted()`** memberi error yang jelas (`NotFittedError`) jika `transform()` dipanggil sebelum `fit()`.
4. **`fit_transform()` tersedia gratis** dari `TransformerMixin`, tanpa kita menulisnya.

## 5. Pipeline

**Pipeline** menggabungkan beberapa transformer dan satu estimator akhir menjadi satu objek.

- `pipe.fit(X, y)` memanggil `fit_transform()` pada tiap transformer, lalu `fit()` pada estimator akhir.
- `pipe.predict(X)` memanggil `transform()` pada tiap transformer, lalu `predict()` pada estimator akhir.

Secara matematis, dengan transformer $T_1, \ldots, T_K$ dan estimator $E$:

$$\hat{y} = E\left(T_K\left(\ldots T_2\left(T_1(\mathbf{X})\right)\right)\right)$$

Manfaatnya: urutan preprocessing selalu sama untuk data latih dan uji, **mencegah data leakage**, dan seluruh alur bisa disimpan sebagai satu file.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=5,
                           n_informative=3, n_redundant=1,
                           random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

pipe.fit(X_train, y_train)

print(f"Akurasi data latih: {pipe.score(X_train, y_train):.4f}")
print(f"Akurasi data uji  : {pipe.score(X_test, y_test):.4f}")
print("Langkah pipeline  :", [name for name, _ in pipe.steps])

Akurasi data latih: 0.9267
Akurasi data uji  : 0.8800
Langkah pipeline  : ['scaler', 'classifier']


**Penjelasan hasil:** `score()` pada classifier mengembalikan **akurasi** (proporsi prediksi yang benar). Selisih akurasi latih dan uji yang kecil menandakan model tidak mengalami overfitting berat. (Angka persisnya bisa sedikit berbeda tergantung versi library.)

Saat `pipe.score(X_test, y_test)` dipanggil, scaler hanya melakukan `transform()` dengan mean dan std dari data latih, tidak di-`fit` ulang. Inilah alasan pipeline otomatis menjaga aturan train/test.

**Di dunia produksi:** pipeline bisa disimpan sebagai satu objek (`joblib.dump(pipe, "model.pkl")`), sehingga preprocessing ikut terbawa dan tidak ada ketidakcocokan versi antara saat training dan serving.

In [15]:
# Menyimpan dan memuat kembali pipeline
import joblib

joblib.dump(pipe, "model.pkl")
loaded = joblib.load("model.pkl")
print("Akurasi uji model hasil load:", round(loaded.score(X_test, y_test), 4))

Akurasi uji model hasil load: 0.88


**Penjelasan hasil:** model yang dimuat ulang memberi skor yang sama persis, karena scaler dan classifier tersimpan bersama.

### Catatan tentang MLOps

Pipeline hanyalah langkah awal. **MLOps** memperluasnya ke seluruh siklus hidup model:

- **Continuous Training (CT):** melatih ulang model otomatis saat ada data baru atau performa turun.
- **CI/CD:** menguji kode dan kualitas data, lalu men-deploy model tanpa langkah manual.
- **Monitoring:** memantau distribusi prediksi dan *data drift* untuk mendeteksi penurunan performa.

Model tidak akan bagus selamanya karena distribusi data berubah seiring waktu. Pipeline membantu karena seluruh alur dari preprocessing sampai prediksi terbungkus dalam satu artefak yang bisa diberi versi.

## 6. Atribut dan Method Umum

Setelah `fit()`, parameter hasil belajar tersimpan sebagai **atribut berakhiran garis bawah** (`_`), misalnya `coef_` dan `intercept_`. Konvensi ini menandakan atribut tersebut baru ada setelah model dilatih.

Untuk model linear, prediksinya:

$$\hat{y} = \mathbf{w}^T \mathbf{x} + b = \sum_{j=1}^{p} w_j x_j + b$$

`score()` mengembalikan metrik default:
- **Classifier:** akurasi = prediksi benar / total prediksi.
- **Regressor:** $R^2 = 1 - \dfrac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$

In [16]:
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression().fit(X, y)

print("Koefisien (w):", model.coef_)
print("Intercept (b):", model.intercept_)
print(f"R-squared    : {model.score(X, y):.6f}")

# Verifikasi manual untuk X = 3
y_hat = model.coef_[0] * 3 + model.intercept_
print(f"\nPrediksi manual X=3: {model.coef_[0]:.2f} * 3 + {model.intercept_:.2f} = {y_hat:.2f}")
print("Nilai asli y untuk X=3:", y[2])

Koefisien (w): [0.95]
Intercept (b): 0.04999999999999938
R-squared    : 0.980978

Prediksi manual X=3: 0.95 * 3 + 0.05 = 2.90
Nilai asli y untuk X=3: 3.0


**Penjelasan hasil:** model belajar $\hat{y} = 0.95x + 0.05$. Nilai $R^2 \approx 0.981$ berarti sekitar **98.1% variasi** pada $y$ dijelaskan oleh hubungan linear dengan $x$. Nilainya belum 1 karena titik $(4, 3.5)$ sedikit menyimpang dari tren lurus.

**Peringatan:** $R^2$ tidak pernah turun saat fitur ditambah, walaupun fitur itu hanya noise. Untuk model dengan banyak fitur, gunakan **adjusted $R^2$**:

$$R^2_{\text{adj}} = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

dengan $n$ jumlah sampel dan $p$ jumlah fitur. Rumus ini memberi penalti pada kompleksitas model.

## 7. Hyperparameter Tuning

**Hyperparameter** adalah pengaturan yang kita tentukan *sebelum* training dan tidak dipelajari dari data, misalnya `n_estimators` (jumlah pohon) pada random forest, `C` (kekuatan regularisasi) pada regresi logistik, atau `max_depth` pada decision tree.

Ada dua pendekatan:
1. **Manual:** `set_params()` dan `get_params()`.
2. **Otomatis:** `GridSearchCV` dan `RandomizedSearchCV`.

### 7.1 Manual: `set_params()` dan `get_params()`

In [17]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.set_params(n_estimators=100, max_depth=10, random_state=42)

print("Seluruh parameter:")
for key, value in sorted(rf.get_params().items()):
    print(f"  {key}: {value}")

Seluruh parameter:
  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 10
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 100
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False


**Penjelasan hasil:** `get_params()` menampilkan semua hyperparameter beserta nilainya. Beberapa yang penting:

- **`n_estimators=100`:** jumlah pohon. Lebih banyak pohon umumnya lebih stabil tetapi lebih lama dilatih.
- **`max_depth=10`:** kedalaman maksimum tiap pohon. Berfungsi sebagai regularisasi agar pohon tidak menghafal data latih.
- **`max_features='sqrt'`** (default): tiap split hanya mempertimbangkan $\sqrt{p}$ fitur sehingga pohon-pohon menjadi lebih beragam (berkurang korelasinya).
- **`criterion='gini'`** (default): ukuran kemurnian node.

$$G = 1 - \sum_{c=1}^{C} p_c^2$$

$p_c$ adalah proporsi kelas $c$ di node tersebut. Node murni punya $G=0$, sedangkan node biner 50/50 punya $G=0.5$.

- **`random_state=42`:** menjamin hasil bisa direproduksi.

### 7.2 Otomatis: `GridSearchCV`

`GridSearchCV` mencoba **semua kombinasi** nilai hyperparameter yang kita daftar, mengevaluasi tiap kombinasi dengan *cross-validation*, lalu memilih yang terbaik. Cocok untuk ruang parameter yang kecil.

In [18]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, None],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
)
grid.fit(X_train, y_train)

print("Parameter terbaik   :", grid.best_params_)
print(f"Skor CV terbaik     : {grid.best_score_:.4f}")
print(f"Skor pada data uji  : {grid.score(X_test, y_test):.4f}")

Parameter terbaik   : {'max_depth': 5, 'n_estimators': 100}
Skor CV terbaik     : 0.9000
Skor pada data uji  : 0.8600


**Penjelasan hasil:** ada 2 x 3 = 6 kombinasi, masing-masing dievaluasi dengan 5-fold CV (total 30 kali training). `best_params_` adalah kombinasi dengan rata-rata skor CV tertinggi. Setelah itu `GridSearchCV` melatih ulang model terbaik pada seluruh data latih, sehingga `grid.score(...)` langsung bisa dipakai pada data uji.

### 7.3 Otomatis: `RandomizedSearchCV`

`RandomizedSearchCV` hanya mencoba sejumlah kombinasi yang **diambil acak** (`n_iter`). Cocok bila ruang parameter besar atau waktu terbatas.

In [19]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    "n_estimators": randint(20, 200),
    "max_depth": randint(2, 12),
}

rand = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=8,
    cv=5,
    random_state=42,
)
rand.fit(X_train, y_train)

print("Parameter terbaik  :", rand.best_params_)
print(f"Skor CV terbaik    : {rand.best_score_:.4f}")
print(f"Skor pada data uji : {rand.score(X_test, y_test):.4f}")

Parameter terbaik  : {'max_depth': 8, 'n_estimators': 199}
Skor CV terbaik    : 0.9067
Skor pada data uji : 0.8800


**Penjelasan hasil:** hanya 8 kombinasi acak yang dicoba, padahal ruang pencariannya jauh lebih besar. Pendekatan ini sering menemukan hasil yang hampir sama bagusnya dengan grid search dengan biaya komputasi lebih kecil.

**Tuning di dalam Pipeline:** nama parameter ditulis `namalangkah__parameter` (dua garis bawah).

In [20]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

grid = GridSearchCV(pipe, {"clf__C": [0.01, 0.1, 1, 10]}, cv=5)
grid.fit(X_train, y_train)

print("C terbaik:", grid.best_params_)
print(f"Skor CV  : {grid.best_score_:.4f}")

C terbaik: {'clf__C': 0.01}
Skor CV  : 0.9133


**Penjelasan hasil:** `clf__C` berarti "parameter `C` milik langkah bernama `clf`". Karena scaler berada di dalam pipeline, ia di-fit ulang hanya pada data latih tiap fold CV, sehingga tidak terjadi data leakage selama tuning.

## 8. Metadata: Tags dan Metadata Routing

scikit-learn memakai **metadata** untuk (a) menjelaskan kemampuan estimator dan (b) meneruskan informasi tambahan seperti `sample_weight` melalui pipeline atau cross-validation.

### 8.1 Estimator Tags

Setiap estimator punya **tags** yang menjelaskan kemampuannya, misalnya apakah mendukung input sparse atau butuh input non-negatif. Tags berfungsi sebagai "kontrak" antara estimator dan infrastruktur scikit-learn.

In [21]:
lr = LogisticRegression()
tags = lr.__sklearn_tags__()

print("Estimator   :", type(lr).__name__)
print("estimator_type      :", tags.estimator_type)
print("requires_fit        :", tags.requires_fit)
print("non_deterministic   :", tags.non_deterministic)
print("input_tags.sparse   :", tags.input_tags.sparse)
print("array_api_support   :", tags.array_api_support)

Estimator   : LogisticRegression
estimator_type      : classifier
requires_fit        : True
non_deterministic   : False
input_tags.sparse   : True
array_api_support   : False


**Penjelasan hasil:** `LogisticRegression` bertipe classifier, perlu `fit()` sebelum dipakai, dan mendukung input sparse. Dalam pemakaian sehari-hari kita jarang menyentuh tags secara langsung, tetapi saat membuat custom estimator kita mungkin perlu mengubah tags agar sesuai dengan kemampuan estimator kita.

### 8.2 Metadata Routing

Masalahnya: bagaimana mengirim `sample_weight` ke classifier di dalam pipeline, tetapi tidak ke scaler? Dengan **metadata routing**, kita menyatakan secara eksplisit komponen mana yang menerima metadata.

- **Router** (`Pipeline`, `GridSearchCV`): meneruskan metadata ke langkah yang tepat.
- **Consumer** (estimator, scorer): memakai metadata tersebut.

In [22]:
import sklearn

# Fitur ini harus diaktifkan secara eksplisit
sklearn.set_config(enable_metadata_routing=True)

rng = np.random.RandomState(0)
sample_weight = rng.uniform(0.5, 2.0, size=len(y_train))

pipe = Pipeline([
    ("scaler", StandardScaler().set_fit_request(sample_weight=False)),  # scaler: abaikan
    ("clf", LogisticRegression(max_iter=1000)
            .set_fit_request(sample_weight=True)),                       # clf: terima
])

# sample_weight diteruskan hanya ke langkah yang memintanya (clf)
pipe.fit(X_train, y_train, sample_weight=sample_weight)
print("Berhasil fit dengan sample_weight lewat pipeline.")
print(f"Akurasi uji: {pipe.score(X_test, y_test):.4f}")

# Kembalikan ke pengaturan awal
sklearn.set_config(enable_metadata_routing=False)

Berhasil fit dengan sample_weight lewat pipeline.
Akurasi uji: 0.8800


**Penjelasan hasil:** `set_fit_request(sample_weight=True)` menandai bahwa langkah `clf` menerima `sample_weight`, sedangkan `set_fit_request(sample_weight=False)` pada scaler menyatakan secara eksplisit bahwa scaler mengabaikannya. Pengaturan eksplisit ini wajib: jika salah satu dibiarkan kosong, scikit-learn akan melempar `UnsetMetadataPassedError` karena `StandardScaler.fit()` sebenarnya juga mampu memakai `sample_weight`. Kegunaan umum: data tidak seimbang atau sampel dengan tingkat kepercayaan berbeda.

## 9. Best Practices

1. **Pakai API yang seragam.** Pola `fit` lalu `predict`/`transform` berlaku untuk semua objek.
2. **Preprocessing sebelum modeling.** Skala, encoding, dan imputasi data sebelum dilatih.
3. **Gunakan pipeline.** Mencegah data leakage, menjaga reproducibility, dan menghasilkan satu artefak untuk deployment.
4. **Validasi dengan cross-validation.** Jangan menilai model hanya dari data latih.
5. **Tuning secara sistematis.** `GridSearchCV` untuk ruang kecil, `RandomizedSearchCV` untuk ruang besar.

In [23]:
# Contoh cross-validation sederhana pada pipeline
from sklearn.model_selection import cross_val_score

# Buat ulang data klasifikasi (variabel X dan y sebelumnya dipakai untuk data regresi)
X, y = make_classification(n_samples=200, n_features=5,
                           n_informative=3, n_redundant=1,
                           random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

scores = cross_val_score(pipe, X, y, cv=5)
print("Skor tiap fold  :", scores.round(4))
print(f"Rata-rata +/- std: {scores.mean():.4f} +/- {scores.std():.4f}")

Skor tiap fold  : [0.875 0.875 0.95  0.9   0.925]
Rata-rata +/- std: 0.9050 +/- 0.0292


**Penjelasan hasil:** `cross_val_score` membagi data menjadi 5 bagian, melatih 5 kali (tiap kali satu bagian jadi data uji), dan memberi 5 skor. Rata-rata dan simpangan bakunya memberi gambaran performa dan stabilitas model yang lebih jujur daripada satu kali train/test split.

## Ringkasan Akhir Chapter 1

| Konsep | Method / Fitur Utama | Peran |
|:--|:--|:--|
| **Estimator** | `fit()`, `predict()`, `fit_predict()` | Belajar dari data dan memprediksi |
| **Transformer** | `fit()`, `transform()`, `fit_transform()` | Memproses dan mengubah data |
| **Custom component** | `BaseEstimator` + mixin | Membuat komponen sendiri yang kompatibel |
| **Pipeline** | `fit()`, `predict()`, `score()` | Merangkai langkah menjadi satu alur |
| **Atribut dan evaluasi** | `coef_`, `intercept_`, `score()` | Memeriksa parameter hasil belajar dan menilai performa |
| **Hyperparameter** | `get_params()`, `set_params()`, `GridSearchCV`, `RandomizedSearchCV` | Mengatur dan mencari konfigurasi terbaik |
| **Metadata** | `__sklearn_tags__()`, metadata routing | Menjelaskan kemampuan estimator dan meneruskan info tambahan |

**Kesimpulan:** keseragaman API membuat scikit-learn produktif. Entah membangun regresi linear sederhana atau pipeline ensemble dengan transformer kustom, antarmukanya tetap sama.